# Granite-CLM-Preview-3B — Lossless CLM Lift & Release

This notebook lifts `ibm-granite/granite-3.1-3b-a800m-base` into the canonical CLM MoE substrate, verifies byte/forward/router/generation parity, records release metrics and provenance, publishes the verified bundle under `archelabs-org/native-clm-v0/granite-clm-preview-3b/`, and pushes lightweight evidence back to GitHub.

**Release boundary:** no training or weight mutation occurs. Expert-slice addresses are deterministic mutation coordinates only; this release does not claim that pretrained Experts are Cells, nor that safe evolution/composability/replay-free learning is solved.

Requirements: Kaggle Internet **ON**, a GPU accelerator (T4-class 16 GB is expected to fit FP16 parity), and Kaggle Secrets named `HF_TOKEN` and `GITHUB_TOKEN`. Publication is fail-closed: Hugging Face upload only runs after every release gate passes.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

BRANCH = "codex/granite-clm-preview-3b-release"
REPO = Path("/kaggle/working/mini-cells")
WORK = Path("/kaggle/working/granite-clm-preview-3b-work")
OUT = REPO / "artifacts/releases/granite-clm-preview-3b"
HF_REPO = "archelabs-org/native-clm-v0"
HF_SUBDIR = "granite-clm-preview-3b"

def run(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), check=True, **kwargs)

if not (REPO / ".git").exists():
    run(["git", "clone", "--branch", BRANCH, "https://github.com/ArcheLabs/mini-cells.git", REPO])
os.chdir(REPO)
run(["git", "fetch", "origin"])
run(["git", "checkout", BRANCH])
run(["git", "pull", "--ff-only", "origin", BRANCH])
run([sys.executable, "-m", "pip", "install", "-e", ".[dev,lm]"])
run([sys.executable, "-m", "pip", "install", "huggingface_hub>=0.28", "accelerate>=0.30"])

import torch
print("branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."


In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["GITHUB_TOKEN"] = secrets.get_secret("GITHUB_TOKEN")
assert os.environ["HF_TOKEN"], "Missing Kaggle Secret: HF_TOKEN"
assert os.environ["GITHUB_TOKEN"], "Missing Kaggle Secret: GITHUB_TOKEN"
print("Secrets loaded (values not printed).")


In [ ]:
# Optional preflight: resolve the current upstream revision before the release run.
from huggingface_hub import HfApi
MODEL_ID = "ibm-granite/granite-3.1-3b-a800m-base"
info = HfApi().model_info(MODEL_ID)
print(json.dumps({
    "source_model": MODEL_ID,
    "resolved_revision": info.sha,
    "hf_target": f"{HF_REPO}/{HF_SUBDIR}",
}, indent=2))


In [ ]:
# Lossless lift -> verification -> metric/provenance recording -> Hugging Face publication.
# The script refuses to upload if any release gate fails.
run([
    sys.executable,
    "scripts/research/run_granite_clm_preview_3b_release.py",
    "--model-id", MODEL_ID,
    "--hf-repo", HF_REPO,
    "--hf-subdir", HF_SUBDIR,
    "--work-dir", WORK,
    "--output-dir", OUT,
    "--device", "cuda",
    "--dtype", "float16",
    "--tolerance", "1e-5",
    "--copy-mode", "hardlink",
    "--publish-hf",
])


In [ ]:
# Render the durable release evidence.
metrics = json.loads((OUT / "metrics.json").read_text())
provenance = json.loads((OUT / "provenance.json").read_text())
parity = json.loads((OUT / "parity_report.json").read_text())
hf_publish = json.loads((OUT / "hf_publish.json").read_text())

summary = {
    "status": metrics["status"],
    "source_revision": provenance["source"]["revision"],
    "hf_target": provenance["hf_target"],
    "hf_commit": hf_publish.get("commit_oid"),
    "manifest_identity_sha256": provenance["manifest_identity_sha256"],
    "layers": metrics["model"]["num_hidden_layers"],
    "experts": metrics["model"]["num_local_experts"],
    "top_k": metrics["model"]["num_experts_per_tok"],
    "expert_address_count": metrics["conversion"]["expert_address_count"],
    "max_abs_logit_error": parity.get("max_abs_logit_error"),
    "max_abs_router_error": parity.get("max_abs_router_error"),
    "router_topk_identity": parity.get("gates", {}).get("router_topk_identity"),
    "greedy_token_identity": parity.get("gates", {}).get("greedy_token_identity"),
    "runner_seconds": metrics["runtime"]["runner_seconds"],
}
print(json.dumps(summary, indent=2))
assert metrics["status"] == "PASS"
assert hf_publish.get("published") is True
print("\n--- RESULTS.md ---\n")
print((OUT / "RESULTS.md").read_text())


In [ ]:
# Push only lightweight release evidence to the dedicated GitHub branch.
run([
    sys.executable,
    "scripts/research/publish_granite_clm_preview_3b.py",
    "--branch", BRANCH,
])
print("Granite-CLM-Preview-3B: Hugging Face bundle and GitHub evidence published.")
